# 🐜 SPA V8 – Sparse Pheromone Attention
## A new attention mechanism inspired by ant colony optimization

**Author:** Independent researcher  
**Concept:** Developed with AI assistance  
**Hardware:** Trained on Google Colab Free T4 (15.6 GB VRAM)

---

### Core Idea / Kernidee

**EN:** Standard Transformers compute attention over ALL tokens (O(n²)).  
SPA selects only k tokens per query using three strategies:
- **local_k** – always attend to the nearest neighbors (local context)
- **learned_k** – globally learned important tokens (via router)
- **explore_k** – random exploration during training only (epsilon-greedy)

A **pheromone buffer** reinforces frequently used attention paths and lets unused ones decay — like ant colonies finding optimal routes.  
**Auto-Tau** is a learnable temperature that self-regulates exploration vs. exploitation.

**DE:** Standard Transformer berechnet Attention über ALLE Token (O(n²)).  
SPA wählt nur k Token pro Query mit drei Strategien:
- **local_k** – immer die nächsten Nachbarn (lokaler Kontext)
- **learned_k** – global gelernte wichtige Token (via Router)
- **explore_k** – zufällige Erkundung nur beim Training (Epsilon-Greedy)

Ein **Pheromone-Buffer** verstärkt häufig genutzte Attention-Pfade und lässt ungenutzte zerfallen — wie Ameisenkolonien optimale Routen finden.  
**Auto-Tau** ist eine lernbare Temperatur die Exploration vs. Exploitation selbst reguliert.

---

### Results / Ergebnisse

| Model | Parameters | Dataset | Steps | PPL | Training Time | Hardware |
|-------|-----------|---------|-------|-----|---------------|----------|
| SPA V8 | 5M | TinyStories 50MB | 10k | 12.7 | 23 min | 1x T4 |
| SPA V8 | 20M | TinyStories 50MB | 10k | 9.06 | 77 min | 1x T4 |
| SPA V8 | 40M | TinyStories 50MB | 10k | 7.12 | 154 min | 1x T4 |
| SPA V8 | 40M | WikiText-103 200MB | 10k | ~55 | 155 min | 1x T4 |
| Transformer-XL* | 41M | WikiText-103 full | 4M | 18.3 | days | 16x V100 |

*For reference only — completely different compute budget*

---
### Just hit Run All / Einfach Run All drücken 🚀


In [ ]:
# ============================================================
# CELL 1: GPU Check & Imports
# EN: Checks if GPU is available and imports required libraries
# DE: Prüft ob GPU verfügbar ist und importiert benötigte Bibliotheken
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time
import os

print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA available / CUDA verfügbar: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device / Verwende: {device}')


In [ ]:
# ============================================================
# CELL 2: Install Dependencies
# EN: Installs required packages
# DE: Installiert benötigte Pakete
# ============================================================

!pip install ninja tokenizers datasets -q


In [ ]:
# ============================================================
# CELL 3: Custom CUDA Kernels (Inference only / nur Inferenz)
# EN: Compiles optimized CUDA kernels for fast inference on T4.
#     Training uses PyTorch autograd fallback (gradients needed).
#     If CUDA is not available, PyTorch fallback is used for both.
# DE: Kompiliert optimierte CUDA Kernels für schnelle Inferenz auf T4.
#     Training nutzt PyTorch Autograd Fallback (Gradienten benötigt).
#     Ohne CUDA wird PyTorch Fallback für beides genutzt.
# ============================================================

from torch.utils.cpp_extension import load_inline

# V5: Sparse dot-product attention with shared memory tiling
# Computes Q·K only for selected k token pairs (not all n²)
CUDA_V5_SRC = """
#include <torch/extension.h>
#include <cuda_runtime.h>

__global__ void sparse_dot_product_tiled_kernel(
    float* logits, const float* q, const float* k,
    const int64_t* topk_indices,
    const int B, const int H, const int T, const int D, const int K)
{
    int b = blockIdx.z;
    int h = blockIdx.y;
    int t_query = blockIdx.x * blockDim.x + threadIdx.x;
    if (t_query < T) {
        extern __shared__ float s_query[];
        for (int d = 0; d < D; d++) {
            s_query[threadIdx.x * D + d] = q[((b * H + h) * T + t_query) * D + d];
        }
        __syncthreads();
        for (int k_idx = 0; k_idx < K; k_idx++) {
            int t_key = topk_indices[b * T * K + t_query * K + k_idx];
            float score = 0.0f;
            for (int d = 0; d < D; d++) {
                score += s_query[threadIdx.x * D + d] * k[((b * H + h) * T + t_key) * D + d];
            }
            logits[((b * H + h) * T + t_query) * K + k_idx] = score;
        }
    }
}

torch::Tensor sparse_attention_v5_tiled(
    torch::Tensor q, torch::Tensor k, torch::Tensor topk_indices)
{
    auto B = q.size(0); auto H = q.size(1);
    auto T = q.size(2); auto D = q.size(3);
    auto K = topk_indices.size(2);
    auto logits = torch::empty({B, H, T, K}, q.options());
    const int threads = 128;
    const dim3 blocks((T + threads - 1) / threads, H, B);
    sparse_dot_product_tiled_kernel<<<blocks, threads, threads * D * sizeof(float)>>>(
        logits.data_ptr<float>(), q.data_ptr<float>(), k.data_ptr<float>(),
        topk_indices.data_ptr<int64_t>(), B, H, T, D, K);
    return logits;
}
"""

# V7: Fused value aggregation
# Aggregates values weighted by attention scores for selected k tokens
CUDA_V7_SRC = """
#include <torch/extension.h>

__global__ void fused_value_aggregation_v7_kernel(
    float* out, const float* attn, const float* v,
    const int64_t* topk,
    const int B, const int H, const int T, const int D, const int K)
{
    int t_query = blockIdx.x * blockDim.x + threadIdx.x;
    int d       = blockIdx.y * blockDim.y + threadIdx.y;
    int bh      = blockIdx.z;
    if (t_query < T && d < D) {
        int b = bh / H; int h = bh % H;
        float sum = 0.0f;
        for (int k_idx = 0; k_idx < K; k_idx++) {
            int t_key = topk[b * T * K + t_query * K + k_idx];
            sum += attn[((b * H + h) * T + t_query) * K + k_idx]
                 * v[((b * H + h) * T + t_key) * D + d];
        }
        out[((b * H + h) * T + t_query) * D + d] = sum;
    }
}

torch::Tensor sparse_value_aggregation_v7(
    torch::Tensor attn, torch::Tensor v, torch::Tensor topk)
{
    auto B = v.size(0); auto H = v.size(1);
    auto T = v.size(2); auto D = v.size(3);
    auto K = topk.size(2);
    auto out = torch::empty({B, H, T, D}, v.options());
    dim3 threads(16, 16);
    dim3 blocks((T+threads.x-1)/threads.x, (D+threads.y-1)/threads.y, B*H);
    fused_value_aggregation_v7_kernel<<<blocks, threads>>>(
        out.data_ptr<float>(), attn.data_ptr<float>(), v.data_ptr<float>(),
        topk.data_ptr<int64_t>(), B, H, T, D, K);
    return out;
}
"""

spa_ops_v5 = None
spa_ops_v7 = None

if torch.cuda.is_available():
    os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'
    if 'CUDA_HOME' not in os.environ:
        os.environ['CUDA_HOME'] = '/usr/local/cuda'

    cuda_flags = ['-O3', '-lineinfo', '-gencode=arch=compute_75,code=sm_75']
    print('Compiling SPA CUDA Kernels for T4 (sm_75)...')

    build_dir = os.path.join(os.getcwd(), 'cuda_extension_build')
    os.makedirs(build_dir, exist_ok=True)

    spa_v5_build_dir = os.path.join(build_dir, 'spa_v5_sm75_build')
    os.makedirs(spa_v5_build_dir, exist_ok=True)
    spa_v7_build_dir = os.path.join(build_dir, 'spa_v7_sm75_build')
    os.makedirs(spa_v7_build_dir, exist_ok=True)

    spa_ops_v5 = load_inline(
        name='spa_v5_sm75',
        cpp_sources='torch::Tensor sparse_attention_v5_tiled(torch::Tensor q, torch::Tensor k, torch::Tensor topk_indices);',
        cuda_sources=CUDA_V5_SRC,
        functions=['sparse_attention_v5_tiled'],
        with_cuda=True,
        extra_cuda_cflags=cuda_flags,
        verbose=False,
        build_directory=spa_v5_build_dir
    )
    spa_ops_v7 = load_inline(
        name='spa_v7_sm75',
        cpp_sources='torch::Tensor sparse_value_aggregation_v7(torch::Tensor attn, torch::Tensor v, torch::Tensor topk);',
        cuda_sources=CUDA_V7_SRC,
        functions=['sparse_value_aggregation_v7'],
        with_cuda=True,
        extra_cuda_cflags=cuda_flags,
        verbose=False,
        build_directory=spa_v7_build_dir
    )
    print('✓ CUDA Kernels active! (V5 Tiled Attention + V7 Fused Value Aggregation)')
else:
    print('No CUDA → PyTorch Fallback active for both training and inference')


In [ ]:
# ============================================================
# CELL 4: Helper Functions + SPA V8 Model Architecture
# EN: Core helper functions and the SPA model definition
# DE: Hilfsfunktionen und die SPA Modell-Definition
# ============================================================

# ── Helper: PyTorch fallback for value gathering ──────────────────────────
def _topk_gather(values, topk_indices):
    B, H, T, D = values.shape
    K = topk_indices.size(-1)
    b_idx = torch.arange(B, device=values.device)[:, None, None, None]
    h_idx = torch.arange(H, device=values.device)[None, :, None, None]
    t_idx = topk_indices[:, None, :, :].expand(B, H, T, K)
    return values[b_idx, h_idx, t_idx, :]

def sparse_attention_impl(q, k, topk_indices):
    """Uses CUDA kernel for inference, PyTorch autograd for training."""
    if spa_ops_v5 is not None and q.is_cuda and not q.requires_grad:
        return spa_ops_v5.sparse_attention_v5_tiled(q, k, topk_indices)
    gathered_k = _topk_gather(k, topk_indices)
    return (q.unsqueeze(3) * gathered_k).sum(dim=-1)

def sparse_value_aggregation_impl(attn, v, topk_indices):
    """Uses CUDA kernel for inference, PyTorch autograd for training."""
    if spa_ops_v7 is not None and v.is_cuda and not v.requires_grad:
        return spa_ops_v7.sparse_value_aggregation_v7(attn, v, topk_indices)
    gathered_v = _topk_gather(v, topk_indices)
    return (attn.unsqueeze(-1) * gathered_v).sum(dim=-2)

def safe_topk_indices(scores, k):
    k = min(k, scores.size(-1))
    if k <= 0:
        return torch.zeros(*scores.shape[:-1], 1, dtype=torch.long, device=scores.device)
    return torch.topk(scores, k=k, dim=-1).indices

# ── Auto-Tau: self-regulating temperature ─────────────────────────────────
def tau_regularization_unweighted(model, target=50.0):
    """Pushes tau towards target value / Schiebt tau Richtung Zielwert."""
    tau = model.current_tau()
    return ((tau - target) / 99.0).pow(2).mean()

def tau_reg_auto_weight(base_weight, ce_loss, reg_unweighted,
                        target_ratio=0.02, min_factor=0.25, max_factor=4.0):
    """Dynamically adjusts tau regularization weight / Passt Tau-Gewicht dynamisch an."""
    if reg_unweighted.item() < 1e-8:
        return base_weight
    desired = target_ratio * ce_loss.item()
    current = base_weight * reg_unweighted.item()
    factor = desired / (current + 1e-8)
    factor = max(min_factor, min(max_factor, factor))
    return base_weight * factor

# ── Pheromone Update ──────────────────────────────────────────────────────
def apply_pheromone_update(pheromone_layer, pheromone_decay, pheromone_alpha,
                           attn, combined_tau_idx, T, n_heads):
    """
    EN: Reinforces frequently used attention paths, lets unused ones decay.
        Like ant pheromones: strong paths get stronger, weak paths disappear.
    DE: Verstärkt häufig genutzte Attention-Pfade, lässt ungenutzte zerfallen.
        Wie Ameisenpheromone: starke Pfade werden stärker, schwache verschwinden.
    """
    with torch.no_grad():
        pheromone = pheromone_layer[:, :T, :T]
        attn_mean = attn.mean(dim=0)
        idx = combined_tau_idx[0].unsqueeze(0).expand(n_heads, -1, -1)
        deposit = torch.zeros_like(pheromone)
        deposit.scatter_add_(dim=-1, index=idx, src=attn_mean)
        pheromone *= (1.0 - pheromone_decay)  # decay: unused paths fade
        pheromone += deposit                   # reinforce: used paths grow
        pheromone.clamp_(0.0, 5.0)

# ── SPA V8 Model ──────────────────────────────────────────────────────────
class SPA_V8_Model(nn.Module):
    def __init__(self,
                 vocab_size,
                 embed_dim=256,      # EN: Embedding dimension | DE: Einbettungsdimension
                 num_heads=8,        # EN: Number of attention heads | DE: Anzahl Attention-Köpfe
                 n_layers=6,         # EN: Number of transformer layers | DE: Anzahl Transformer-Layer
                 k=48,               # EN: Total sparse tokens per query | DE: Gesamte Sparse-Token pro Query
                 max_seq_len=512,    # EN: Maximum sequence length | DE: Maximale Sequenzlänge
                 local_k=26,         # EN: Always attend to nearest N neighbors | DE: Immer die N nächsten Nachbarn
                 explore_k=4,        # EN: Random exploration tokens (training only) | DE: Zufällige Erkundungs-Token (nur Training)
                 tau_min=1.0,        # EN: Minimum temperature (sharp attention) | DE: Minimale Temperatur (scharfe Attention)
                 tau_max=100.0,      # EN: Maximum temperature (soft attention) | DE: Maximale Temperatur (weiche Attention)
                 tau_init=10.0,      # EN: Initial temperature | DE: Start-Temperatur
                 dropout=0.1,        # EN: Dropout rate | DE: Dropout-Rate
                 pheromone_decay=0.02 # EN: How fast unused paths fade (0=never, 1=instant) | DE: Wie schnell ungenutzte Pfade zerfallen
                 ):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.n_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.k = k
        self.n_layers = n_layers
        self.max_seq_len = max_seq_len
        self.local_k = min(local_k, k)
        self.explore_k = max(0, min(explore_k, k - self.local_k))
        self.pheromone_decay = pheromone_decay
        self.pheromone_alpha = 0.15  # EN: Pheromone influence strength | DE: Pheromone-Einfluss-Stärke
        self.tau_min = tau_min
        self.tau_max = tau_max

        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(max_seq_len, embed_dim)

        # EN: One pheromone buffer per layer — stored in model state_dict automatically
        # DE: Ein Pheromone-Buffer pro Layer — wird automatisch in state_dict gespeichert
        for i in range(n_layers):
            self.register_buffer(f'pheromone_{i}', torch.zeros(self.n_heads, max_seq_len, max_seq_len))

        # EN: Learnable log-tau parameter — auto-regulates during training
        # DE: Lernbarer log-tau Parameter — reguliert sich selbst während Training
        eps = 1e-4
        init_ratio = (tau_init - tau_min) / (tau_max - tau_min)
        init_ratio = min(max(init_ratio, eps), 1.0 - eps)
        self.log_tau = nn.Parameter(torch.full((1, 1, k), math.log(init_ratio / (1.0 - init_ratio))))

        # EN: Global router: learns which positions are globally important
        # DE: Globaler Router: lernt welche Positionen global wichtig sind
        self.global_router = nn.Linear(embed_dim, max_seq_len)
        self.dropout = nn.Dropout(dropout)
        self.emb_dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList([
            nn.ModuleList([
                nn.LayerNorm(embed_dim),                    # pre-norm 1
                nn.Linear(embed_dim, 3 * embed_dim),        # QKV projection
                nn.Linear(embed_dim, embed_dim),            # output projection
                nn.LayerNorm(embed_dim),                    # pre-norm 2
                nn.Sequential(                              # FFN
                    nn.Linear(embed_dim, 4 * embed_dim),
                    nn.GELU(),
                    nn.Dropout(dropout),
                    nn.Linear(4 * embed_dim, embed_dim)
                )
            ]) for _ in range(n_layers)
        ])

        self.ln_f = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size)

    def current_tau(self):
        """Returns current tau value (learnable temperature)."""
        return self.tau_min + (self.tau_max - self.tau_min) * torch.sigmoid(self.log_tau)

    def get_pheromone(self, layer_id):
        return getattr(self, f'pheromone_{layer_id}')

    def _build_sparse_indices(self, x):
        """
        EN: Builds sparse attention index set per query token:
            local_k  = nearest previous tokens (always included)
            learned_k = globally important tokens (learned by router)
            explore_k = random tokens for exploration (training only!)
        DE: Baut Sparse-Attention-Indizes pro Query-Token:
            local_k   = nächste vorherige Token (immer dabei)
            learned_k = global wichtige Token (vom Router gelernt)
            explore_k = zufällige Token zur Erkundung (nur Training!)
        """
        B, T, _ = x.shape
        device = x.device
        parts = []

        if self.local_k > 0:
            positions = torch.arange(T, device=device)
            query_pos = positions.view(1, T, 1)
            local_offsets = torch.arange(1, self.local_k + 1, device=device).view(1, 1, self.local_k)
            local_idx = (query_pos - local_offsets).clamp(min=0).expand(B, -1, -1)
            parts.append(local_idx)

        if self.k - self.local_k > 0:
            causal_mask = torch.triu(torch.ones(T, T, device=device), diagonal=1).bool()
            router_logits = self.global_router(x)[:, :, :T]
            router_logits = router_logits.masked_fill(causal_mask, float('-inf'))
            gk = self.k - self.local_k
            learned_k = max(1, gk // 2)
            learned_idx = safe_topk_indices(router_logits, learned_k)
            global_parts = [learned_idx]

            # EN: Explorer tokens: random paths, TRAINING ONLY (disabled at inference)
            # DE: Explorer-Token: zufällige Pfade, NUR TRAINING (bei Inferenz deaktiviert)
            if self.training and self.explore_k > 0 and gk - learned_k > 0:
                random_scores = torch.rand(B, T, T, device=device)
                random_scores = random_scores.masked_fill(causal_mask, -1.0)
                explore_idx = safe_topk_indices(random_scores, gk - learned_k)
                global_parts.append(explore_idx)

            global_idx = torch.cat(global_parts, dim=-1)
            if global_idx.size(-1) > gk:
                global_idx = global_idx[..., :gk]
            elif global_idx.size(-1) < gk:
                pad = torch.zeros(B, T, gk - global_idx.size(-1), device=device, dtype=torch.long)
                global_idx = torch.cat([global_idx, pad], dim=-1)
            parts.append(global_idx)

        combined = torch.cat(parts, dim=-1) if len(parts) > 1 else parts[0]
        return combined.long()

    def forward(self, idx):
        B, T = idx.shape
        device = idx.device

        x = self.token_emb(idx) + self.pos_emb(torch.arange(T, device=device))
        x = self.emb_dropout(x)

        tau_base = self.current_tau()
        tau_scale = (40.0 / (tau_base + 1e-8)).clamp(0.3, 3.5)

        for layer_id, (ln1, qkv_l, proj, ln2, mlp) in enumerate(self.layers):
            pheromone_buf = self.get_pheromone(layer_id)
            combined_tau_idx = self._build_sparse_indices(x)

            h = ln1(x)
            qkv = qkv_l(h).reshape(B, T, 3, self.n_heads, self.head_dim)
            q = qkv[:, :, 0].transpose(1, 2).contiguous()
            k = qkv[:, :, 1].transpose(1, 2).contiguous()
            v = qkv[:, :, 2].transpose(1, 2).contiguous()

            # Sparse Q·K (CUDA V5 or PyTorch fallback)
            raw_logits = sparse_attention_impl(q, k, combined_tau_idx) * (self.head_dim ** -0.5)

            # Add pheromone bias — reinforces frequently used paths
            pheromone = pheromone_buf[:, :T, :T]
            pheromone_bias = pheromone.unsqueeze(0).expand(B, -1, -1, -1)
            idx_expanded = combined_tau_idx.unsqueeze(1).expand(B, self.n_heads, T, -1)
            gathered_pheromone = torch.gather(pheromone_bias, dim=-1, index=idx_expanded)

            logits = raw_logits + self.pheromone_alpha * gathered_pheromone
            logits = logits * tau_scale.unsqueeze(1)

            attn = F.softmax(logits, dim=-1)
            attn = self.dropout(attn)

            # Update pheromone buffer
            apply_pheromone_update(
                pheromone_buf, self.pheromone_decay, self.pheromone_alpha,
                attn, combined_tau_idx, T, self.n_heads
            )

            # Sparse value aggregation (CUDA V7 or PyTorch fallback)
            out = sparse_value_aggregation_impl(attn, v, combined_tau_idx)
            x = x + self.dropout(proj(out.transpose(1, 2).reshape(B, T, self.embed_dim)))
            x = x + mlp(ln2(x))

        return self.head(self.ln_f(x)), tau_base, combined_tau_idx

kernel_mode = 'CUDA V5+V7' if (spa_ops_v5 and spa_ops_v7) else 'PyTorch Fallback'
print(f'✓ SPA V8 Model ready! ({kernel_mode})')


In [ ]:
# ============================================================
# CELL 5: Dataset & Tokenizer
# EN: Loads dataset and trains BPE tokenizer.
#     Uses streaming to avoid RAM overflow.
#     Switch between TinyStories and WikiText-103 here.
# DE: Lädt Dataset und trainiert BPE Tokenizer.
#     Nutzt Streaming um RAM-Überlauf zu vermeiden.
#     Hier zwischen TinyStories und WikiText-103 wechseln.
# ============================================================

from datasets import load_dataset
from tokenizers import ByteLevelBPETokenizer

# ── DATASET CHOICE / DATASET AUSWAHL ─────────────────────────────────────
# EN: Change DATASET to switch between datasets
# DE: DATASET ändern um zwischen Datasets zu wechseln
#
# Options / Optionen:
#   "tinystories"  → roneneldan/TinyStories  (simple, fast, good for first tests)
#   "wikitext103"  → wikitext-103-raw-v1     (complex, real benchmark)

DATASET = "wikitext103"   # ← change here / hier ändern

# EN: How much data to load (in characters). More = better quality but more RAM/time
# DE: Wie viel Daten laden (in Zeichen). Mehr = bessere Qualität aber mehr RAM/Zeit
MAX_CHARS = 200_000_000   # 200MB — reduce to 100_000_000 if RAM crash / reduzieren bei RAM-Absturz

# ── TOKENIZER VOCAB SIZE ──────────────────────────────────────────────────
# EN: 8000 is a good balance. Larger = better quality but slower training.
# DE: 8000 ist ein guter Kompromiss. Grösser = bessere Qualität aber langsameres Training.
VOCAB_SIZE_TARGET = 8000

# ─────────────────────────────────────────────────────────────────────────

def stream_dataset(dataset_name, max_chars):
    """Generator that streams text from dataset / Generator der Text streamt."""
    if dataset_name == "tinystories":
        ds = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
        for ex in ds:
            if ex["text"].strip():
                yield ex["text"]
                max_chars -= len(ex["text"])
                if max_chars <= 0:
                    break
    elif dataset_name == "wikitext103":
        ds = load_dataset("wikitext", "wikitext-103-raw-v1", split="train", streaming=True)
        for ex in ds:
            if ex["text"].strip():
                yield ex["text"]
                max_chars -= len(ex["text"])
                if max_chars <= 0:
                    break

# Step 1: Train tokenizer (streaming, low RAM)
print(f"Training BPE Tokenizer on {DATASET} ({MAX_CHARS//1_000_000}MB)...")
tokenizer = ByteLevelBPETokenizer()
tokenizer.train_from_iterator(
    stream_dataset(DATASET, MAX_CHARS),
    vocab_size=VOCAB_SIZE_TARGET,
    min_frequency=2,
    special_tokens=["<pad>", "<unk>", "<bos>", "<eos>"]
)
tokenizer.save_model(".", "spa_tokenizer")
VOCAB_SIZE = tokenizer.get_vocab_size()
encode = lambda s: tokenizer.encode(s).ids
decode = lambda l: tokenizer.decode(l)
print(f"✓ Tokenizer ready! Vocab: {VOCAB_SIZE}")

# Step 2: Encode data (second stream pass, low RAM)
print(f"Encoding dataset...")
all_ids = []
chars = 0
for text in stream_dataset(DATASET, MAX_CHARS):
    all_ids.extend(encode(text))
    chars += len(text)

data = torch.tensor(all_ids, dtype=torch.long)
del all_ids  # free RAM / RAM freigeben
print(f'✓ Dataset ready! Vocab: {VOCAB_SIZE} | Tokens: {len(data):,}')


In [ ]:
# ============================================================
# CELL 6: Model Configuration / Modell-Konfiguration
# EN: Configure model size here. Larger = better quality but more VRAM/time.
# DE: Modellgrösse hier konfigurieren. Grösser = bessere Qualität aber mehr VRAM/Zeit.
#
# Tested configurations / Getestete Konfigurationen:
#   5M params:  embed_dim=192, n_layers=4  → ~23 min, 32MB VRAM
#   20M params: embed_dim=384, n_layers=8  → ~77 min, 5.2GB VRAM
#   40M params: embed_dim=512, n_layers=10 → ~154 min, 11GB VRAM  ← DEFAULT
# ============================================================

model = SPA_V8_Model(
    vocab_size    = VOCAB_SIZE,
    embed_dim     = 512,    # EN: Larger = more capacity | DE: Grösser = mehr Kapazität
    num_heads     = 8,      # EN: Must divide embed_dim evenly | DE: Muss embed_dim teilen
    n_layers      = 10,     # EN: Depth of the network | DE: Tiefe des Netzwerks
    k             = 48,     # EN: Sparse tokens per query (total) | DE: Sparse-Token pro Query (gesamt)
    max_seq_len   = 512,    # EN: Context window size | DE: Kontextfenstergrösse
    local_k       = 26,     # EN: Always attend to last N tokens | DE: Immer letzte N Token beachten
    explore_k     = 4,      # EN: Random explorer tokens (training only) | DE: Zufällige Explorer-Token (nur Training)
    tau_init      = 10.0,   # EN: Initial attention temperature (auto-adjusts) | DE: Start-Attention-Temperatur (passt sich an)
    dropout       = 0.1,    # EN: Dropout rate for regularization | DE: Dropout-Rate zur Regularisierung
    pheromone_decay = 0.02  # EN: Decay rate for unused paths (0=never, 1=instant) | DE: Zerfallsrate ungenutzter Pfade
).to(device)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'✓ Model parameters / Modellparameter: {total_params:.2f}M')
if torch.cuda.is_available():
    print(f'  GPU memory after model / GPU-Speicher nach Modell: {torch.cuda.memory_allocated() / 1e6:.1f} MB')


In [ ]:
# ============================================================
# CELL 7: Training Setup
# EN: Configure training hyperparameters here.
# DE: Trainings-Hyperparameter hier konfigurieren.
# ============================================================

BATCH_SIZE       = 8      # EN: Samples per gradient step | DE: Samples pro Gradient-Schritt
BLOCK_SIZE       = 512    # EN: Sequence length (must match max_seq_len) | DE: Sequenzlänge (muss max_seq_len entsprechen)
TRAIN_STEPS      = 10000  # EN: Total training steps | DE: Gesamte Trainings-Schritte
CHECKPOINT_EVERY = 1000   # EN: Save checkpoint every N steps | DE: Checkpoint alle N Schritte speichern

# ── Auto-Tau Parameters ───────────────────────────────────────────────────
# EN: These control how tau (attention temperature) self-regulates.
#     Usually no need to change these.
# DE: Diese steuern wie tau (Attention-Temperatur) sich selbst reguliert.
#     Normalerweise muss man das nicht ändern.
TAU_TARGET              = 40.0  # EN: Target tau value | DE: Ziel-Tau-Wert
TAU_REG_WEIGHT          = 1e-4  # EN: Base regularization weight | DE: Basis-Regularisierungsgewicht
TAU_REG_WARMUP_STEPS    = 2000  # EN: Steps before tau reg kicks in | DE: Schritte bevor Tau-Reg einsetzt
AUTO_TAU_TUNE           = True  # EN: Automatically adjust tau weight | DE: Tau-Gewicht automatisch anpassen
TAU_REG_TARGET_RATIO    = 0.02
TAU_REG_AUTO_MIN_FACTOR = 0.25
TAU_REG_AUTO_MAX_FACTOR = 4.0
TAU_REG_EMA_BETA        = 0.95
tau_reg_weight_ema      = TAU_REG_WEIGHT

# ── Data Split ────────────────────────────────────────────────────────────
n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - BLOCK_SIZE, (BATCH_SIZE,))
    xb = torch.stack([d[j:j+BLOCK_SIZE] for j in ix]).to(device)
    yb = torch.stack([d[j+1:j+BLOCK_SIZE+1] for j in ix]).to(device)
    return xb, yb

@torch.no_grad()
def estimate_val_loss(eval_iters=50):
    model.eval()
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
        xb, yb = get_batch('val')
        logits, _, _ = model(xb)
        losses[k] = F.cross_entropy(logits.view(-1, VOCAB_SIZE), yb.view(-1)).item()
    model.train()
    return losses.mean().item()

# EN: Separate learning rates: main params use 3e-4, tau uses 1e-3 (faster adaptation)
# DE: Getrennte Lernraten: Hauptparameter 3e-4, Tau 1e-3 (schnellere Anpassung)
optimizer = torch.optim.AdamW([
    {'params': [p for n, p in model.named_parameters() if 'log_tau' not in n], 'lr': 3e-4},
    {'params': model.log_tau, 'lr': 1e-3}
], weight_decay=0.05)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TRAIN_STEPS)

print(f'✓ Training setup ready: {TRAIN_STEPS} steps, batch={BATCH_SIZE}')
print(f'  Train tokens: {len(train_data):,} | Val tokens: {len(val_data):,}')


In [ ]:
# ============================================================
# CELL 8: Training Loop
# EN: Main training loop. Checkpoints are saved automatically.
#     You can interrupt and resume from any checkpoint.
# DE: Haupt-Trainings-Schleife. Checkpoints werden automatisch gespeichert.
#     Du kannst unterbrechen und von jedem Checkpoint weitermachen.
# ============================================================

model.train()
metrics = {'step': [], 'train_loss': [], 'val_loss': [], 'perplexity': [], 'tau': []}

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

training_start = time.time()
step_start     = time.time()
tokens_since_last = 0

print(f'Starting SPA V8 Training ({TRAIN_STEPS} Steps, Batch={BATCH_SIZE})...\n')

for step in range(TRAIN_STEPS + 1):
    xb, yb = get_batch('train')
    tokens_since_last += BATCH_SIZE * BLOCK_SIZE

    logits, tau_vals, _ = model(xb)
    ce_loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), yb.view(-1))

    # Auto-Tau regularization
    tau_reg_raw = tau_regularization_unweighted(model, target=TAU_TARGET)
    tau_reg_weight_auto = tau_reg_auto_weight(
        base_weight=TAU_REG_WEIGHT, ce_loss=ce_loss, reg_unweighted=tau_reg_raw,
        target_ratio=TAU_REG_TARGET_RATIO,
        min_factor=TAU_REG_AUTO_MIN_FACTOR, max_factor=TAU_REG_AUTO_MAX_FACTOR,
    ) if AUTO_TAU_TUNE else TAU_REG_WEIGHT

    tau_reg_weight_ema = TAU_REG_EMA_BETA * tau_reg_weight_ema + (1.0 - TAU_REG_EMA_BETA) * tau_reg_weight_auto
    tau_reg_weight_cur = min(1.0, step / max(1, TAU_REG_WARMUP_STEPS)) * tau_reg_weight_ema
    loss = ce_loss + tau_reg_weight_cur * tau_reg_raw

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()

    if step % 500 == 0:
        val_loss    = estimate_val_loss()
        ppl         = math.exp(val_loss)
        current_tau = model.current_tau().mean().item()
        elapsed     = time.time() - training_start
        tok_per_sec = tokens_since_last / (time.time() - step_start + 1e-8)
        step_start  = time.time()
        tokens_since_last = 0

        metrics['step'].append(step)
        metrics['train_loss'].append(ce_loss.item())
        metrics['val_loss'].append(val_loss)
        metrics['perplexity'].append(ppl)
        metrics['tau'].append(current_tau)

        lr_current = optimizer.param_groups[0]['lr']
        print(f'Step {step:5d} | Loss: {ce_loss.item():.4f} | Val: {val_loss:.4f} | '
              f'PPL: {ppl:.2f} | Tau: {current_tau:.1f} | '
              f'LR: {lr_current:.2e} | Tok/s: {tok_per_sec:.0f} | '
              f'Zeit: {elapsed:.0f}s')

    if step % CHECKPOINT_EVERY == 0 and step > 0:
        path = f'spa_v8_checkpoint_step{step}.pth'
        torch.save({
            'step': step,
            'model_state_dict': model.state_dict(),      # includes log_tau and pheromones!
            'optimizer_state_dict': optimizer.state_dict(),
            'metrics': metrics
        }, path)
        print(f'  → Checkpoint saved: {path}')

# Final results
if torch.cuda.is_available():
    peak_mem = torch.cuda.max_memory_allocated() / 1e6
    print(f'\n{"="*55}')
    print(f'SPA V8 FINAL RESULTS')
    print(f'{"="*55}')
    print(f'Final Val Loss:       {metrics["val_loss"][-1]:.4f}')
    print(f'Final Val Perplexity: {metrics["perplexity"][-1]:.2f}')
    print(f'Training time:        {(time.time()-training_start)/60:.1f} Minutes')
    print(f'GPU Peak Memory:      {peak_mem:.1f} MB')
    print(f'{"="*55}')


In [ ]:
# ============================================================
# CELL 9: Load Checkpoint (optional)
# EN: Load a previously saved checkpoint to continue training
#     or to generate text from a specific step.
# DE: Gespeicherten Checkpoint laden um Training fortzusetzen
#     oder Text von einem bestimmten Schritt zu generieren.
# ============================================================

CHECKPOINT_PATH = 'spa_v8_checkpoint_step8000.pth'  # ← change step number / Schritt-Nummer ändern

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
# EN: log_tau and pheromones are in model_state_dict — restored automatically!
# DE: log_tau und Pheromone sind in model_state_dict — werden automatisch wiederhergestellt!

if 'metrics' in checkpoint:
    metrics = checkpoint['metrics']

current_tau = model.current_tau().mean().item()
print(f'✓ Loaded checkpoint: {CHECKPOINT_PATH}')
print(f'  Resumed from step: {checkpoint["step"]}')
print(f'  Tau restored to:   {current_tau:.1f}')


In [ ]:
# ============================================================
# CELL 10: Text Generation / Text-Generierung
# EN: Generate text from a prompt. Explorer tokens are automatically
#     disabled at inference (model.eval() sets self.training=False).
# DE: Text aus einem Prompt generieren. Explorer-Token werden bei
#     Inferenz automatisch deaktiviert (model.eval() setzt self.training=False).
# ============================================================

def top_p_sampling(logits, p=0.9):
    """Nucleus sampling — keeps only top-p probability mass."""
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
    sorted_indices_to_remove = cumulative_probs > p
    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
    sorted_indices_to_remove[..., 0] = 0
    indices_to_remove = sorted_indices[sorted_indices_to_remove]
    logits[indices_to_remove] = float('-inf')
    return logits

# ── Generation Parameters / Generierungs-Parameter ───────────────────────
PROMPT      = "Once upon a time"   # ← change prompt here / Prompt hier ändern
GEN_LENGTH  = 500    # EN: Tokens to generate | DE: Zu generierende Token
TEMPERATURE = 0.65   # EN: Lower = more focused, Higher = more creative | DE: Tiefer = fokussierter, Höher = kreativer
TOP_P       = 0.9    # EN: Nucleus sampling threshold | DE: Nucleus-Sampling-Schwellwert
T_CTX       = 512    # EN: Context window (must match max_seq_len) | DE: Kontextfenster

model.eval()  # EN: Disables dropout AND explorer tokens! | DE: Deaktiviert Dropout UND Explorer-Token!
if torch.cuda.is_available():
    torch.cuda.empty_cache()

idx = torch.tensor(encode(PROMPT), dtype=torch.long).unsqueeze(0).to(device)
gen_start = time.time()
print(f'Generating {GEN_LENGTH} tokens (Temp: {TEMPERATURE}, T_CTX: {T_CTX})...\n')

with torch.no_grad():
    for i in range(GEN_LENGTH):
        idx_cond = idx[:, -T_CTX:]
        logits, _, _ = model(idx_cond)
        logits = logits[0, -1, :] / TEMPERATURE
        logits = top_p_sampling(logits, p=TOP_P)
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_token.unsqueeze(0)], dim=1)

gen_time = time.time() - gen_start
print(f'Generation time: {gen_time:.1f}s | Tokens/sec: {GEN_LENGTH/gen_time:.1f}')
print(f'\n--- Result SPA V8 ---\n')
print(decode(idx[0].tolist()))


In [ ]:
# ============================================================
# CELL 11: Training Curves / Trainingskurven
# ============================================================

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(metrics['step'], metrics['train_loss'], color='salmon', label='Train Loss')
axes[0].plot(metrics['step'], metrics['val_loss'],   color='steelblue', label='Val Loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('Steps')
axes[0].legend()

axes[1].plot(metrics['step'], metrics['perplexity'], color='green')
axes[1].set_title('Perplexity (Val)')
axes[1].set_xlabel('Steps')

axes[2].plot(metrics['step'], metrics['tau'], color='orange')
axes[2].set_title('Tau (Auto-regulated temperature)')
axes[2].set_xlabel('Steps')

plt.tight_layout()
plt.savefig('spa_v8_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: spa_v8_training_curves.png')


In [ ]:
# ============================================================
# CELL 12: Pheromone Visualization / Pheromone-Visualisierung
# EN: Shows which attention paths the model has reinforced.
#     Bright = frequently used, Dark = rarely used.
#     Diagonal = local patterns, Off-diagonal = long-range dependencies.
# DE: Zeigt welche Attention-Pfade das Modell verstärkt hat.
#     Hell = häufig genutzt, Dunkel = selten genutzt.
#     Diagonale = lokale Muster, Off-Diagonale = Langstrecken-Abhängigkeiten.
# ============================================================

!pip install seaborn -q
import seaborn as sns
import matplotlib.pyplot as plt

def plot_pheromone_layers(model, seq_len=128):
    n    = model.n_layers
    cols = 3
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 5))
    axes = axes.flatten()

    for i in range(n):
        phero = model.get_pheromone(i).detach().cpu().numpy()
        avg   = phero.mean(axis=0)[:seq_len, :seq_len]
        sns.heatmap(avg, cmap='hot', ax=axes[i], cbar=True)
        axes[i].set_title(f'Layer {i}')

    for i in range(n, len(axes)):
        axes[i].set_visible(False)

    plt.suptitle('SPA V8 – Pheromone per Layer (avg over heads)', fontsize=14)
    plt.tight_layout()
    plt.savefig('spa_v8_pheromone_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✓ Saved: spa_v8_pheromone_heatmap.png')

plot_pheromone_layers(model, seq_len=128)
